In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import networkx as nx
from networkx.algorithms import smallworld
import random
from collections import Counter
from scipy.spatial import cKDTree

import pandas as pd

In [3]:
from src.neuron_population import NeuronPopulation
from src.connectome import Connectome
from src.overhead import Simulation
from src.neuron_templates import neuron_type_IZ
from src.network_grower import *
from src.network_generators import *
from src.neuron_type_distributor import *
from src.network_weight_distributor import *
from src.external_inputs import *

In [4]:
weight_scale = 1.0
g = 1.2984752590298583 * 60

J_I = weight_scale * g
J_E = weight_scale
delay_mean_E = 10.0
delay_std_E = delay_mean_E * 0.3
delay_mean_I = 1.5
delay_std_I = delay_mean_I * 0.3
v_ext = 0.45527031369449

excitatory_type = "ss4"
inhibitory_type = "b"

In [5]:
seed = 1234

## Topology

In [6]:
G = nx.DiGraph()

rng = np.random.default_rng(seed)


n_neurons = int(1000 * 1.0)

I_percent = 0.2

n_excitatory = int(n_neurons * (1 - I_percent))

density = 0.1
    
# Add n_neurons nodes
for i in range(n_neurons):
    G.add_node(i)

# Assign excitatory and inhibitory neurons
# excitatory_nodes = random.sample(range(n_neurons), n_excitatory)

for i in range(n_excitatory):
        G.nodes[i]['inhibitory'] = False
        G.nodes[i]['ntype'] = excitatory_type
        G.nodes[i]['layer'] = 0

for i in range(n_excitatory, n_neurons):
        G.nodes[i]['inhibitory'] = True
        G.nodes[i]['ntype'] = inhibitory_type
        G.nodes[i]['layer'] = 0

# For each node, draw m outgoing edges to random nodes
n_out = int(n_neurons * density)
for i in range(n_neurons):
    targets = rng.choice(range(n_neurons), n_out, replace=False)
    for target in targets:
        if G.nodes[i]['inhibitory']:
            weight = J_I
            delay = max(0.1, rng.normal(delay_mean_I, delay_std_I))
        else:
            weight = J_E
            delay = max(0.1, rng.normal(delay_mean_E, delay_std_E))
        G.add_edge(i, target, weight=weight, distance=delay)

## Trial analysis

In [7]:
# load df from csv nsga2_run14/pareto_front.csv
df = pd.read_csv('nsga2_run15/pareto_front.csv')

df

,number,state,objective.persistence,objective.asynchrony,objective.regime,objective.transmitter_balance,param.A_AMPA,param.A_GABA_A,param.A_GABA_B,param.A_NMDA,...,metric.topology_n_edges,metric.topology_n_nodes,metric.topology_out_degree_mean,metric.topology_out_degree_std,metric.topology_reciprocity,metric.trial_seed,metric.voltage_rest_var_post_exc_mV2,metric.voltage_rest_var_post_exc_mV2_std,metric.voltage_rest_var_post_inh_mV2,metric.voltage_rest_var_post_inh_mV2_std
0,359,COMPLETE,1.000000,1.962502,-0.963155,0.586312,0.037637,0.001107,0.000352,0.018118,...,100000.0,1000.0,100.0,0.0,0.0987,364474,428.791503,0.0,1547.832976,0.0
1,342,COMPLETE,1.000000,1.897217,-17.589489,0.607521,0.037637,0.046852,0.000352,0.004019,...,100000.0,1000.0,100.0,0.0,0.0987,347321,377.681810,0.0,1556.831694,0.0
2,347,COMPLETE,1.000000,1.838249,-0.903779,0.699844,0.037637,0.001107,0.000352,0.002003,...,100000.0,1000.0,100.0,0.0,0.0987,352366,438.908936,0.0,1425.057404,0.0
3,248,COMPLETE,1.000000,1.832277,-0.843212,0.676215,0.037637,0.001107,0.000352,0.002003,...,100000.0,1000.0,100.0,0.0,0.0987,252475,438.867204,0.0,1205.249914,0.0
4,277,COMPLETE,1.000000,1.812467,-0.641617,0.609562,0.011895,0.000941,0.000128,0.003884,...,100000.0,1000.0,100.0,0.0,0.0987,281736,389.827621,0.0,694.820272,0.0
5,233,COMPLETE,1.000000,1.539008,-0.350400,0.345922,0.000222,0.039691,0.000352,0.037936,...,100000.0,1000.0,100.0,0.0,0.0987,237340,137.743913,0.0,554.702419,0.0
6,324,COMPLETE,1.000000,1.501816,-0.293985,0.258741,0.000222,0.001107,0.000352,0.024865,...,100000.0,1000.0,100.0,0.0,0.0987,329159,165.716781,0.0,421.183458,0.0
7,148,COMPLETE,1.000000,1.462019,-0.284233,0.355432,0.000222,0.039691,0.000352,0.020917,...,100000.0,1000.0,100.0,0.0,0.0987,151575,119.958912,0.0,439.898148,0.0
8,114,COMPLETE,1.000000,1.460647,-0.311180,0.355838,0.000222,0.039691,0.000352,0.020917,...,100000.0,1000.0,100.0,0.0,0.0987,117269,126.075374,0.0,506.726435,0.0
9,303,COMPLETE,1.000000,1.427149,-0.483615,0.374386,0.000222,0.039691,0.003241,0.003884,...,100000.0,1000.0,100.0,0.0,0.0987,307970,248.900500,0.0,931.363557,0.0


In [8]:
df.columns

Index(['number', 'state', 'objective.persistence', 'objective.asynchrony',
       'objective.regime', 'objective.transmitter_balance', 'param.A_AMPA',
       'param.A_GABA_A', 'param.A_GABA_B', 'param.A_NMDA',
       ...
       'metric.topology_n_edges', 'metric.topology_n_nodes',
       'metric.topology_out_degree_mean', 'metric.topology_out_degree_std',
       'metric.topology_reciprocity', 'metric.trial_seed',
       'metric.voltage_rest_var_post_exc_mV2',
       'metric.voltage_rest_var_post_exc_mV2_std',
       'metric.voltage_rest_var_post_inh_mV2',
       'metric.voltage_rest_var_post_inh_mV2_std'],
      dtype='object', length=113)

In [13]:
df[df["number"] == 303]["metric.psd_peak_ratio"]

KeyError: 'metric.psd_peak_ratio'

In [10]:
# add sum column of normalize columns 'objective.persistence', 'objective.asynchrony',
 #      'objective.regime', 'objective.transmitter_balance'

# Do standard min-max normalization for each of the objective columns
for col in ['objective.persistence', 'objective.asynchrony', 'objective.regime', 'objective.transmitter_balance']:
    min_val = df[col].min()
    max_val = df[col].max()
    norm_col_name = col + '_norm'
    df[norm_col_name] = (df[col] - min_val) / (max_val - min_val)

# Then sum the normalized columns to get a combined score
df['objective.combined_score'] = (df['objective.persistence_norm'] + 
                                  df['objective.asynchrony_norm'] + 
                                  df['objective.regime_norm'] +
                                  df['objective.transmitter_balance_norm'])

# Sort the DataFrame by the combined score in descending order
df_sorted = df.sort_values(by='objective.combined_score', ascending=False)

In [11]:
df_sorted

,number,state,objective.persistence,objective.asynchrony,objective.regime,objective.transmitter_balance,param.A_AMPA,param.A_GABA_A,param.A_GABA_B,param.A_NMDA,...,metric.trial_seed,metric.voltage_rest_var_post_exc_mV2,metric.voltage_rest_var_post_exc_mV2_std,metric.voltage_rest_var_post_inh_mV2,metric.voltage_rest_var_post_inh_mV2_std,objective.persistence_norm,objective.asynchrony_norm,objective.regime_norm,objective.transmitter_balance_norm,objective.combined_score
2,347,COMPLETE,1.000000,1.838249,-0.903779,0.699844,0.037637,0.001107,0.000352,0.002003,...,352366,438.908936,0.0,1425.057404,0.0,1.000000,0.864398,0.962350,0.978768,3.805516
3,248,COMPLETE,1.000000,1.832277,-0.843212,0.676215,0.037637,0.001107,0.000352,0.002003,...,252475,438.867204,0.0,1205.249914,0.0,1.000000,0.857881,0.965843,0.926920,3.750645
0,359,COMPLETE,1.000000,1.962502,-0.963155,0.586312,0.037637,0.001107,0.000352,0.018118,...,364474,428.791503,0.0,1547.832976,0.0,1.000000,1.000000,0.958926,0.729653,3.688579
4,277,COMPLETE,1.000000,1.812467,-0.641617,0.609562,0.011895,0.000941,0.000128,0.003884,...,281736,389.827621,0.0,694.820272,0.0,1.000000,0.836262,0.977470,0.780667,3.594400
12,119,COMPLETE,0.980037,1.687945,-0.574429,0.333500,0.000222,0.001571,0.000175,0.020917,...,122314,164.630263,0.0,1395.924728,0.0,0.931439,0.700367,0.981346,0.174925,2.788077
5,233,COMPLETE,1.000000,1.539008,-0.350400,0.345922,0.000222,0.039691,0.000352,0.037936,...,237340,137.743913,0.0,554.702419,0.0,1.000000,0.537827,0.994266,0.202182,2.734275
1,342,COMPLETE,1.000000,1.897217,-17.589489,0.607521,0.037637,0.046852,0.000352,0.004019,...,347321,377.681810,0.0,1556.831694,0.0,1.000000,0.928753,0.000000,0.776189,2.704942
7,148,COMPLETE,1.000000,1.462019,-0.284233,0.355432,0.000222,0.039691,0.000352,0.020917,...,151575,119.958912,0.0,439.898148,0.0,1.000000,0.453807,0.998083,0.223050,2.674939
8,114,COMPLETE,1.000000,1.460647,-0.311180,0.355838,0.000222,0.039691,0.000352,0.020917,...,117269,126.075374,0.0,506.726435,0.0,1.000000,0.452309,0.996528,0.223939,2.672777
9,303,COMPLETE,1.000000,1.427149,-0.483615,0.374386,0.000222,0.039691,0.003241,0.003884,...,307970,248.900500,0.0,931.363557,0.0,1.000000,0.415752,0.986583,0.264638,2.666973
